libraries

In [10]:
import sqlite3
import pandas as pd

In [11]:
# Connect to the SQLite database
paytm_payments_database_path = r"D:\nishkama karma\nidhidyasana\Paytm-FinTech-Analytics-AI-Platform\payments_fraud_analytics\paytm_payments.db"
conn = sqlite3.connect(paytm_payments_database_path)

Load merchants.csv, users.csv, and ledger.csv into a normalized SQLite database paytm_payments.db with a schema that has merchants(merchant_id PK, ...), users(user_id PK, signup_date), and transactions(transaction_id PK, user_id FK, merchant_id FK, ...)

In [12]:
# merchants(merchant_id PK, ...)
conn.execute('''CREATE TABLE IF NOT EXISTS merchants (
    merchant_id INTEGER PRIMARY KEY,
    merchant_name TEXT,
    category TEXT,
    region TEXT
)''') 

# users(user_id PK, signup_date)
conn.execute('''CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY,
    signup_date datetime
)''') 

# transactions(transaction_id PK, user_id FK, merchant_id FK, ...)
conn.execute('''CREATE TABLE IF NOT EXISTS transactions (
    transaction_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    merchant_id INTEGER,
    transaction_time datetime,
    amount_inr REAL,
    payment_method TEXT,
    status TEXT,
    risk_score REAL,
    FOREIGN KEY (user_id) REFERENCES users(user_id),
    FOREIGN KEY (merchant_id) REFERENCES merchants(merchant_id)
)''')


Quantify chargeback impact: count of chargeback transactions, unique users affected, total chargeback amount.

In [22]:
query = ''' 
select distinct status from transactions 

'''

result = pd.read_sql_query(query, conn)
print(result) 

       status
0    captured
1      failed
2  chargeback


In [20]:
query = ''' 
select 
count(transaction_id) as chargeback_count,
count(distinct user_id) as unique_users_count,
sum(amount_inr) as total_chargeback_amount
from transactions 
where status = 'chargeback' 
'''

result = pd.read_sql_query(query, conn)
print(result) 

   chargeback_count  unique_users_count  total_chargeback_amount
0                28                  27                    54472


Identify burner accounts: users whose signup_date is less than 30 days before their transaction's transaction_time, restricted to status = 'chargeback'. Make the boundary explicit and unambiguous: 0 <= (transaction_time - signup_date).days < 30 — the signup must be on or before the transaction (never a negative age) and strictly less than 30 days earlier (never exactly 30 days or more). Your query must surface at least the 15 seeded burner-account rows.

In [23]:
query = '''
        select 
        users.user_id
        from transactions 
        inner join users on transactions.user_id = users.user_id 
        where transactions.status = 'chargeback' 
        and julianday(transactions.transaction_time) - julianday(users.signup_date) < 30
        and julianday(transactions.transaction_time) - julianday(users.signup_date) >= 0;
        ''' 

result = pd.read_sql_query(query, conn)
result

,user_id


Detect velocity attacks: users with 3 or more transactions within any 10-minute window. Your query must surface at least the 8 seeded velocity clusters. Grading clarification: your result is correct if, when grouped by user_id and a rounded/floored 10-minute time bucket of transaction_time, all 8 seeded clusters (each identifiable by its victim user_id and the cluster's earliest transaction_time) appear as distinct qualifying groups — graders check for the presence of all 8 seeded (user_id, approximate-cluster-start-time) combinations, NOT an exact row-count or exact-bucket-boundary match, since overlapping 10-minute windows can be grouped multiple reasonable ways.

In [29]:
# users with 3 or more transactions within any 10-minute window. 
# Your query must surface at least the 8 seeded velocity clusters
# Grading clarification: your result is correct if, when grouped by user_id and a rounded/floored 10-minute time bucket of transaction_time, 
# all 8 seeded clusters (each identifiable by its victim user_id and 
# the cluster's earliest transaction_time) appear as distinct qualifying groups — graders check for the presence of all 8 seeded 
# (user_id, approximate-cluster-start-time) combinations, NOT an exact row-count or exact-bucket-boundary match, since overlapping 10-minute
# windows can be grouped multiple reasonable ways.

''' 
  -- understanding the problem: we are looking for users who have made 3 or more transactions within any 10-minute window. 
  -- What i understood :- Users with 3 or more transactios with in any 10-minute window. 
                          -- For above i need to use user table and transaction table 
                          
'''

query = ''' 
select 
user_id,
SUBSTR(transaction_time, 1, 15) || '0:00' as time_bucket,
COUNT(transaction_id) AS transaction_count 
from transactions
group by user_id, time_bucket
having count(transaction_id) >= 3;
'''

result = pd.read_sql_query(query, conn)
result 


,user_id,time_bucket,transaction_count
0,59,09-01-2026 21:00:00,4
1,73,12-01-2026 09:00:00,4
2,154,02-01-2026 22:00:00,4
3,200,01-01-2026 22:00:00,4
4,229,12-01-2026 12:00:00,4
5,287,14-01-2026 14:00:00,4
6,314,02-01-2026 18:00:00,4
7,345,23-01-2026 09:00:00,4
